# PlantDoc AI — Two-Stage Training Orchestrator (Colab)

**Mục tiêu:** Thực hiện chiến lược huấn luyện 2 giai đoạn để thu hẹp khoảng cách miền (domain gap):
1. **Stage 1:** Huấn luyện trên PlantVillage (ảnh sạch) để học kiến thức nền nông nghiệp.
2. **Stage 2:** Fine-tune trên PlantDoc (ảnh thực tế) để tăng khả năng tổng quát hóa.

**Quy trình Workflow chuẩn (Split-First):**
1. **Chuẩn bị Dữ liệu:** Giải nén `extended.zip` vào ổ đĩa nội bộ (local) của Colab để tối ưu tốc độ đọc.
2. **Tạo Split Động:** Tự động tạo phân chia tập dữ liệu (train/val/test) và class mapping TỪ ĐẦU bằng các script có sẵn, trỏ thẳng tới dữ liệu vừa giải nén.
3. **Huấn luyện:** Gọi `scripts/train.py` với cấu hình đã được điều chỉnh tự động cho Colab.
4. **Lưu trữ an toàn:** Lưu toàn bộ Checkpoint, Logs, và Artifacts thẳng lên Google Drive.

> ⚠️ **Gợi ý:** Hãy chắc chắn bạn đã chọn **Runtime → Change runtime type → GPU** trước khi bắt đầu.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Khai báo Cấu hình và Đường dẫn

Thiết lập các biến môi trường và đường dẫn cần thiết cho phiên huấn luyện. Bạn có thể thay đổi `DATASET_ZIP_ON_DRIVE` và `DRIVE_WORK_ROOT` nếu cần.

In [ ]:
import os
from pathlib import Path

# =============================================================================
# [CẤU HÌNH NGƯỜI DÙNG]
# =============================================================================

# 1. Dữ liệu nguồn trên Drive (Chứa các thư mục plantVillage và PlantDoc)
DATASET_ZIP_ON_DRIVE = "/content/drive/MyDrive/AI/PlantDocAI_Colab/extended.zip"

# 2. Thư mục làm việc trên Drive (Nơi lưu Artifacts, Checkpoints, Logs)
DRIVE_WORK_ROOT = "/content/drive/MyDrive/AI/PlantDocAI_Colab"

# 3. Lấy Codebase
CODE_SOURCE = "git"  # "git" (clone từ github) hoặc "drive" (dùng code có sẵn trên Drive)
GIT_URL = "https://github.com/bien3008/PlantDocAI.git"
GIT_BRANCH = "main"
PROJECT_DIR_ON_DRIVE = "/content/drive/MyDrive/AI/PlantDocAI_Colab/PlantDocAI"

# =============================================================================
# [ĐƯỜNG DẪN HỆ THỐNG COLAB]
# =============================================================================
PROJECT_ROOT = "/content/PlantDocAI"
EXTRACT_ROOT = "/content/datasets/extended"
DRIVE_WORK_ROOT = Path(DRIVE_WORK_ROOT)

# Cấu hình huấn luyện gốc
STAGE1_CONFIG = "configs/stage1.yaml"
STAGE2_CONFIG = "configs/stage2.yaml"

print(f"✅ Dữ liệu zip: {DATASET_ZIP_ON_DRIVE}")
print(f"✅ Kết quả sẽ lưu tại: {DRIVE_WORK_ROOT}")

## 3. Thiết lập Môi trường và Codebase

In [ ]:
import shutil

if CODE_SOURCE == "git":
    if os.path.exists(PROJECT_ROOT):
        print(f"🧹 Đang xóa thư mục cũ {PROJECT_ROOT} để clone lại mới...")
        shutil.rmtree(PROJECT_ROOT)
    !git clone --branch {GIT_BRANCH} {GIT_URL} {PROJECT_ROOT}
else:
    PROJECT_ROOT = PROJECT_DIR_ON_DRIVE

%cd {PROJECT_ROOT}

print("\n📦 Cài đặt dependencies (yêu cầu từ requirements.txt)...")
!pip install -q -r requirements.txt
print("✅ Môi trường sẵn sàng.")

## 4. Chuẩn bị Dữ liệu (Giải nén & Tìm kiếm động)

Giải nén file zip vào bộ nhớ SSD nội bộ của Colab để tăng cường tốc độ truy xuất ảnh (I/O) thay vì đọc trực tiếp từ Drive. Sau đó, tự động dò tìm vị trí chính xác của các thư mục dữ liệu.

In [ ]:
import zipfile
from pathlib import Path

if not os.path.exists(EXTRACT_ROOT):
    print(f"📦 Đang giải nén dataset vào {EXTRACT_ROOT}... Quá trình này có thể mất vài phút.")
    os.makedirs(EXTRACT_ROOT, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP_ON_DRIVE, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_ROOT)
    print("✅ Giải nén hoàn tất.")
else:
    print(f"✅ Dữ liệu đã được giải nén sẵn tại {EXTRACT_ROOT}")

def find_data_path(root, folder_name, sub_folder="train"):
    """Dò tìm đường dẫn thư mục để tránh lỗi do cấu trúc file zip bị đóng gói nhiều lớp."""
    matches = list(Path(root).rglob(folder_name))
    if not matches:
        return None
    base_path = matches[0]
    final_path = base_path / sub_folder
    if final_path.is_dir():
        return str(final_path)
    # Fallback kiểm tra cấu trúc bên trong
    class_dirs = [d for d in base_path.iterdir() if d.is_dir() and "___" in d.name]
    if class_dirs:
        return str(base_path)
    return None

# Tìm các thư mục cần thiết
DATA_PV = find_data_path(EXTRACT_ROOT, "plantVillage", "train")
DATA_PD_TRAIN = find_data_path(EXTRACT_ROOT, "PlantDoc_Dataset_master", "train")
DATA_PD_TEST = find_data_path(EXTRACT_ROOT, "PlantDoc_Dataset_master", "test")

if not DATA_PV or not DATA_PD_TRAIN or not DATA_PD_TEST:
    print("❌ LỖI NGHIÊM TRỌNG: Không tìm thấy thư mục dữ liệu PlantVillage hoặc PlantDoc sau khi giải nén!")
    print("Cấu trúc thư mục giải nén:")
    !ls -R {EXTRACT_ROOT} | head -n 30
    raise FileNotFoundError("Không tìm đủ thư mục dữ liệu. Hãy kiểm tra cấu trúc file zip.")

print("\n✅ ĐÃ TÌM THẤY DỮ LIỆU:")
print(f"  - PlantVillage Train : {DATA_PV}")
print(f"  - PlantDoc Train     : {DATA_PD_TRAIN}")
print(f"  - PlantDoc Test      : {DATA_PD_TEST}")

## 5. Tạo Dữ Liệu Phân Chia (Splits) Mới

YÊU CẦU BẮT BUỘC: Không sử dụng splits cũ. Chúng ta sẽ gọi trực tiếp các scripts có sẵn trong dự án (`buildCoreClassMapping.py` và `createTwoStageSplits.py`) và truyền đường dẫn thực tế của Colab.

In [ ]:
import os

SPLIT_OUT_DIR = "/content/splits_two_stage"
CORE_CLASSES_CSV = f"{SPLIT_OUT_DIR}/core_classes.csv"

os.makedirs(SPLIT_OUT_DIR, exist_ok=True)

print("\n======================================================")
print(" [BƯỚC 1] XÂY DỰNG CORE CLASS MAPPING (16 Classes) ")
print("======================================================")
!python scripts/buildCoreClassMapping.py \
    --pvDir "{DATA_PV}" \
    --pdTrainDir "{DATA_PD_TRAIN}" \
    --pdTestDir "{DATA_PD_TEST}" \
    --outDir "{SPLIT_OUT_DIR}"

print("\n======================================================")
print(" [BƯỚC 2] TẠO FILE SPLITS (CSV) CHO TỪNG STAGE ")
print("======================================================")
!python scripts/createTwoStageSplits.py \
    --pvDir "{DATA_PV}" \
    --pdTrainDir "{DATA_PD_TRAIN}" \
    --pdTestDir "{DATA_PD_TEST}" \
    --coreClassesCsv "{CORE_CLASSES_CSV}" \
    --outBase "{SPLIT_OUT_DIR}"


### 5.1 Kiểm tra Split (Sanity Check)

Đảm bảo files phân chia đã được tạo thành công trước khi bắt đầu huấn luyện.

In [ ]:
import sys

required_splits = [
    (f"{SPLIT_OUT_DIR}/stage1", ["train.csv", "val.csv", "test.csv", "classes.csv"]),
    (f"{SPLIT_OUT_DIR}/stage2", ["train.csv", "val.csv", "classes.csv"]),
]

missing = False
for folder, files in required_splits:
    for file in files:
        p = Path(folder) / file
        if not p.exists():
            print(f"❌ Thiếu file: {p}")
            missing = True
        else:
            print(f"✅ Đã tạo file: {p} (Size: {p.stat().st_size} bytes)")

if missing:
    print("\n❌ LỖI: Quá trình tạo splits thất bại. Vui lòng kiểm tra log phía trên.")
    sys.exit(1)
else:
    print("\n✅ TẤT CẢ SPLITS ĐÃ SẴN SÀNG!")

## 6. Điều chỉnh Config cho Colab (Config Patching)

Tạo các file config mới (`_colab.yaml`) dựa trên cấu hình dự án, nhưng trỏ trực tiếp đến `SPLIT_OUT_DIR` (chứa các CSV vừa tạo), `DATA_DIR` (dữ liệu ảnh đã giải nén trên Colab), và `OUTPUT_DIR` (lưu kết quả bền vững trên Drive).

In [ ]:
import yaml

def patch_config(base_cfg_path, new_data_dir, new_split_dir, out_root, stage_name):
    if not os.path.exists(base_cfg_path):
        raise FileNotFoundError(f"Không tìm thấy config gốc: {base_cfg_path}")
        
    with open(base_cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)
    
    # 1. Trỏ dataDir vào thư mục ảnh gốc của Stage đó
    cfg['dataDir'] = new_data_dir
    
    # 2. Trỏ splitDir vào thư mục splits vừa tạo
    cfg['splitDir'] = new_split_dir
    
    # 3. Trỏ outputDir lưu ra Google Drive
    exp_name = cfg['experimentName']
    cfg['outputDir'] = str(DRIVE_WORK_ROOT / "artifacts" / exp_name)
    
    patched_path = base_cfg_path.replace('.yaml', f'_colab_{stage_name}.yaml')
    with open(patched_path, 'w') as f:
        yaml.dump(cfg, f)
        
    return patched_path

S1_PATCHED = patch_config(STAGE1_CONFIG, DATA_PV, f"{SPLIT_OUT_DIR}/stage1", DRIVE_WORK_ROOT, "stage1")
S2_PATCHED = patch_config(STAGE2_CONFIG, DATA_PD_TRAIN, f"{SPLIT_OUT_DIR}/stage2", DRIVE_WORK_ROOT, "stage2")

print("✅ Cấu hình Stage 1:", S1_PATCHED)
print("✅ Cấu hình Stage 2:", S2_PATCHED)

## 7. HUẤN LUYỆN (Train Model)

Sử dụng `scripts/train.py`. Script này hỗ trợ cờ `--resume` (tự động khôi phục từ checkpoint `last.pt` nếu tiến trình bị gián đoạn).

### 🚀 Stage 1: Domain Pretraining (PlantVillage)
Mô hình học kiến thức nền nông nghiệp trên 16 core classes từ ảnh sạch.

In [ ]:
!python scripts/train.py --config {S1_PATCHED} --useGdrive --resume

### 🚀 Stage 2: Fine-tuning (PlantDoc)
Mô hình chuyển giao kiến thức (Transfer Learning) bằng cách lấy trọng số từ `best.pt` của Stage 1, và học cách nhận diện lá bệnh trong môi trường nhiễu thực tế (PlantDoc).

In [ ]:
!python scripts/train.py --config {S2_PATCHED} --useGdrive --resume

## 8. Đánh giá Cuối cùng trên Benchmark (PlantDoc Test)

Sau khi Stage 2 hoàn tất, gọi `scripts/evaluate.py` để kiểm tra độ chính xác trên tập dữ liệu PlantDoc Test (môi trường thực tế 100%).

In [ ]:
import yaml

with open(S2_PATCHED, 'r') as f:
    s2_cfg = yaml.safe_load(f)
MODEL_DIR = s2_cfg['outputDir']

print("======================================================")
print(f" 📊 ĐÁNH GIÁ MÔ HÌNH: {MODEL_DIR}")
print("======================================================")

!python scripts/evaluate.py --modelDir {MODEL_DIR} --split test \
    --dataDir {DATA_PD_TEST} \
    --splitDir {SPLIT_OUT_DIR}/plantdoc_test